This notebook (`main_chainsaw.ipynb`) adapts the [HMM-for-text-decryption](https://github.com/alessimichele/HMM-for-text-decryption) repository for use in DTE-2501 AI Methods and Applications.  

Instead of decryption, the HMM is trained on bigrams (pairs of characters) from the preprocessed *Chainsaw Man* transcripts. The goal is to predict the next bigram from the previous one, and compare the results to a simple Markov chain model.  

Workflow:
- Load bigram states (`hmm_bigram_states.txt`) created from the cleaned dataset into HMM-for-text-decryption repo.  
- Initialize and train the HMM using the Baum–Welch algorithm.  
- Generate new sequences and stitch them back into text.  

In [ ]:
from pathlib import Path
import numpy as np
from collections import Counter

DATA_FILE = Path("texts/hmm_bigram_states.txt")  # din fil
raw = [ln.rstrip("\n\r") for ln in open(DATA_FILE, encoding="utf-8")]

# keep only bigrams (length 2)
bigrams_all = [bg for bg in raw if len(bg) == 2]
freq = Counter(bigrams_all)

# Choose the K most frequent bigrams as alphabet
K = 27
alphabet = [bg for bg, _ in freq.most_common(K)]  # 27 mest frekvente bigrams
alphabet_set = set(alphabet)

# filter the sequence to only include these symbols
bigrams = [bg for bg in bigrams_all if bg in alphabet_set]

sym2id = {s:i for i,s in enumerate(alphabet)}     # id 0..26
id2sym = {i:s for s,i in sym2id.items()}

observed_sequence = np.array([sym2id[s] for s in bigrams], dtype=int)

print("Chosen alphabet (27):", alphabet)
print(f"T = {len(observed_sequence)}, M = {len(alphabet)} (må være 27)")


In [ ]:
import numpy as np

N = 27                      # hidden states
M = 27                      # observation symbols 
rng = np.random.default_rng(0)

# random A (N×N) row-normalized, and uniform pi
A = rng.random((N, N)); A /= A.sum(axis=1, keepdims=True)
pi = np.full(N, 1.0 / N)

# start-emission (N×M) uniform
B0 = np.full((N, M), 1.0 / M)

print("Shapes -> A", A.shape, "pi", pi.shape, "B0", B0.shape)


In [ ]:
from src.HMM_functions import Baum_Welch
import numpy as np
from tqdm import tqdm

# train with progress bar
def train_with_progress(A, B0, pi, observed, iters=60):
    B = B0.copy()
    for _ in tqdm(range(iters), desc="Baum–Welch"):
        B = Baum_Welch(A=A, B_start=B, pi=pi, observed=observed, maxIter=1)
    return B

emission = train_with_progress(A, B0, pi, observed_sequence, iters=10)
print("B shape:", emission.shape)

# Sample from the trained HMM
rng = np.random.default_rng(0)

def sample_hmm(pi, A, B, length=800):
    s = rng.choice(N, p=pi)
    obs = []
    for _ in range(length):
        o = rng.choice(M, p=B[s])   # emit symbol
        obs.append(int(o))
        s = rng.choice(N, p=A[s])   # next state
    return np.array(obs, dtype=int)

gen_ids = sample_hmm(pi, A, emission, length=800)
gen_bigrams = [id2sym[i] for i in gen_ids]

def stitch_bigrams(bgs):
    return "" if not bgs else bgs[0][0] + "".join(bg[1] for bg in bgs)

text = stitch_bigrams(gen_bigrams)
print(text[:600])


In [ ]:
# Sample multiple snippets from the trained HMM
def hmm_generate_snippets(pi, A, B, n_snippets=5, length=500, seed=0):
    rng = np.random.default_rng(seed)
    def sample(length):
        N, M = A.shape[0], B.shape[1]
        s = rng.choice(N, p=pi)
        ids = []
        for _ in range(length):
            o = rng.choice(M, p=B[s])
            ids.append(int(o))
            s = rng.choice(N, p=A[s])
        return ids
    outs = []
    for i in range(n_snippets):
        ids = sample(length)
        bgs = [id2sym[j] for j in ids]
        txt = bgs[0][0] + "".join(bg[1] for bg in bgs)
        outs.append(txt)
    return outs

snips = hmm_generate_snippets(pi, A, emission, n_snippets=5, length=500, seed=42)
for i,s in enumerate(snips, 1):
    print(f"\n=== HMM sample #{i} ===\n{s[:500]}")
    
# save to file
with open("hmm_samples.txt","w",encoding="utf-8") as f:
    for s in snips:
        f.write(s + "\n\n")
print("Wrote hmm_samples.txt")
